# A2 – Odnosi med spremenljivkami

**Razdelek B · vodena demonstracija · Olist**

Nadaljujemo po A1.

Primerjamo Pearsonov in Spearmanov koeficient, povezave narišemo, primerjamo skupine,
preberemo kontingenčne tabele ter poiščemo močno povezane značilke. Na koncu ločimo
opis povezave od vzročne razlage.


## Priprava

Notebook je samostojen in deluje lokalno ter v Google Colabu. Celice izvajaj od
zgoraj navzdol s **Shift + Enter**. Ob menjavi stolpcev ponovi ustrezne naslednje celice.

Prva celica poišče `utils/orodja.py`; v Colabu po potrebi prenese repozitorij in namesti
knjižnice iz `requirements.txt`. Lokalno jih enkrat namestimo po navodilih v README.
Druga celica pokliče skupni `poskrbi_za_podatke()` in vrne mapo `DATA`.
Funkcija uporabi pripravljene ali že prenesene podatke; manjkajoče prenese s Kaggla.
CSV-je nato beremo iz `DATA`, ne glede na mapo, v kateri je zvezek odprt.
Izvornih vrednosti ne popravljamo in rezultatov vaj ne zapisujemo samodejno v datoteke.


In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)


### Skupni podatkovni vhod
`poskrbi_za_podatke()` iz `utils/orodja.py` pripravi skupno mapo podatkov in jo vrne kot `DATA`.

Izvirne tabele Olista beremo iz `DATA / "trgovina"`; raven opazovanja pripravimo v nadaljevanju.


In [ ]:
from utils.orodja import poskrbi_za_podatke
DATA = poskrbi_za_podatke()


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 15)
pd.set_option("display.max_colwidth", 35)
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11})


### En zapis = eno dostavljeno naročilo

Izdelke najprej povzamemo po `order_id`. Vrednost naročila je vsota cen izdelkov
**brez poštnine**, v BRL. Podatke kupca povežemo po `customer_id`.

Za povezave z oceno potrebujemo eno oceno na naročilo. Če obstaja več zapisov,
izberemo **zadnjega po `review_answer_timestamp`**; izenačitve razrešimo po datumu
nastanka in `review_id`. To je izrecna analitična odločitev. V A1 smo ocene
opisovali po zapisih ocen, zato se imenovalec in povzetki lahko razlikujejo.
Neocenjena naročila ostanejo v osnovni tabeli.

Trajanje dostave je čas od nakupa do predaje v dneh. Zamudo primerjamo po
**koledarskem datumu**, ker je obljubljeni datum zapisan ob polnoči. Dostava na
obljubljeni dan je pravočasna. Neznanega datuma ne označimo kot pravočasnega.


In [ ]:
mapa_podatkov = DATA / "trgovina"
narocila = pd.read_csv(mapa_podatkov / "olist_orders_dataset.csv")
izdelki_narocil = pd.read_csv(mapa_podatkov / "olist_order_items_dataset.csv")
kupci = pd.read_csv(
    mapa_podatkov / "olist_customers_dataset.csv",
    dtype={"customer_zip_code_prefix": "string"},
)
vse_ocene = pd.read_csv(mapa_podatkov / "olist_order_reviews_dataset.csv")

display(pd.DataFrame({
    "tabela": ["naročila", "izdelki naročil", "kupci", "zapisi ocen"],
    "vrstice": [len(narocila), len(izdelki_narocil), len(kupci), len(vse_ocene)],
}))


In [ ]:
zneski_narocil = izdelki_narocil.groupby("order_id", as_index=False).agg(
    vrednost_narocila=("price", "sum"), postnina=("freight_value", "sum"),
    stevilo_izdelkov=("order_item_id", "size"), stevilo_prodajalcev=("seller_id", "nunique"),
)
ocene_urejene = vse_ocene.copy()
for stolpec in ["review_answer_timestamp", "review_creation_date"]:
    ocene_urejene[stolpec] = pd.to_datetime(ocene_urejene[stolpec])
zadnje_ocene = (
    ocene_urejene.sort_values(
        ["review_answer_timestamp", "review_creation_date", "review_id"],
        kind="stable", na_position="first",
    ).drop_duplicates("order_id", keep="last")[["order_id", "review_score"]]
    .rename(columns={"review_score": "ocena"})
)
df = (
    narocila.loc[narocila["order_status"].eq("delivered")]
    .merge(zneski_narocil, on="order_id", how="left", validate="one_to_one")
    .merge(kupci[["customer_id", "customer_state"]],
           on="customer_id", how="left", validate="one_to_one")
    .merge(zadnje_ocene, on="order_id", how="left", validate="one_to_one").copy()
)
for stolpec in ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]:
    df[stolpec] = pd.to_datetime(df[stolpec])
df["trajanje_dostave_dni"] = (
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.total_seconds() / 86400
df["zamuda_dni"] = (
    df["order_delivered_customer_date"].dt.normalize()
    - df["order_estimated_delivery_date"].dt.normalize()
).dt.days
df["zamuda"] = pd.Series(pd.NA, index=df.index, dtype="string")
znana_zamuda = df["zamuda_dni"].notna()
df.loc[znana_zamuda, "zamuda"] = np.where(
    df.loc[znana_zamuda, "zamuda_dni"].gt(0), "z zamudo", "pravočasno"
)
df["vrednost_s_postnino"] = df["vrednost_narocila"] + df["postnina"]
assert df["order_id"].is_unique
print(f"Naročil: {len(df):,}; z znano oceno: {df['ocena'].count():,}")
print(f"Neznana zamuda: {df['zamuda'].isna().sum()}")
display(df[["vrednost_narocila", "postnina", "stevilo_izdelkov",
            "trajanje_dostave_dni", "zamuda", "ocena"]].head())


## 1. Kako izberemo orodje?


| Tip para | Orodje | Kaj vedno izpišemo? |
|---|---|---|
| Numerična + numerična | Pearson, Spearman, razsevni diagram | Število znanih parov |
| Kategorija + numerična | `groupby`, povzetki po skupinah | Velikost skupine in število znanih meritev |
| Kategorija + kategorija | `pd.crosstab` | Števila in imenovalec deležev |
| Več numeričnih stolpcev | Korelacijska matrika | Izbor meritev in število znanih parov |

Pri vsaki nalogi izpustimo le zapise brez vrednosti, potrebnih za **to vprašanje**.
Celotne tabele ne čistimo s splošnim `dropna()`.


## 2. Pearson in Spearman: dve številki in slika

Primerjamo `vrednost_narocila` in `postnina`. Pearson opisuje linearno, Spearman
monotono povezavo na rangih. Razlika je povod za ogled grafa, ne sama po sebi dokaz
nelinearnosti ali vpliva osamelcev. Poskusi tudi trajanje dostave in oceno.

Koeficienta uporabljata vse znane pare. Na obeh grafih je isti reproducibilni vzorec;
desni graf le omeji osi. Korelacije ne preračunamo na omejenem prikazu.


In [ ]:
x = 'vrednost_narocila'
y = 'postnina'
pari = df[[x, y]].dropna()
display(pd.Series({
    "znanih parov": len(pari),
    "Pearson": pari.corr(method="pearson").loc[x, y],
    "Spearman": pari.corr(method="spearman").loc[x, y],
}).to_frame("vrednost").round(3))

# Koeficienta uporabljata vse znane pare; graf reproducibilen vzorec.
vzorec = pari.sample(n=min(2500, len(pari)), random_state=42)
fig, osi = plt.subplots(1, 2, figsize=(11, 4))
for os in osi:
    os.scatter(vzorec[x], vzorec[y], s=9, alpha=0.25)
    os.set(xlabel=x, ylabel=y)
osi[0].set_title("Celoten razpon vzorca")
osi[1].set_title("Povečan prikaz: osi med 1. in 99. percentilom")
osi[1].set_xlim(pari[x].quantile(0.01), pari[x].quantile(0.99))
osi[1].set_ylim(pari[y].quantile(0.01), pari[y].quantile(0.99))
fig.tight_layout()
plt.show()


**Vprašanja skupini:** ali imajo naročila podobne vrednosti enako poštnino? Kaj
pokaže povečan prikaz? Ali bi en sam koeficient zadostoval za opis celotnega razpona?


## 3. Česa koeficient ne pokaže: ocena po trajanju dostave

En koeficient ne pokaže pragov ali lokalnih razlik. Tudi pri neničelnem koeficientu
lahko razredi razkrijejo več o poteku povezave. Koeficient blizu nič pa še ne izključi
drugih, nemonotonih oblik povezave.

Ocena 1–5 je ordinalna. Povprečje uporabimo kot praktičen skupinski povzetek, ob
njem izpišemo mediano in velikost skupine; pozneje pogledamo celotne frekvence.


In [ ]:
pari_dostava = df[["trajanje_dostave_dni", "ocena"]].dropna().copy()
print(f"Znanih parov: {len(pari_dostava):,}")
display(pari_dostava.corr(method="pearson").round(3))
display(pari_dostava.corr(method="spearman").round(3))
pari_dostava["razred_dostave"] = pd.cut(
    pari_dostava["trajanje_dostave_dni"], bins=[0, 3, 7, 14, 21, 30, 60, np.inf],
    labels=["do 3", "3–7", "7–14", "14–21", "21–30", "30–60", "nad 60"],
)
po_trajanju = pari_dostava.groupby("razred_dostave", observed=True)["ocena"].agg(
    narocila="size", povprecna_ocena="mean", mediana_ocene="median"
)
display(po_trajanju.round(3))
fig, os = plt.subplots()
po_trajanju["povprecna_ocena"].plot.bar(ax=os)
os.set(xlabel="Trajanje dostave v dneh; desna meja je vključena", ylabel="Povprečna ocena",
       ylim=(1, 5), title="Potek povezave po razredih")
fig.tight_layout()
plt.show()


**Vprašanja:** kje se povprečna ocena najbolj spremeni? Kako velike so skupine?

Meje razredov so naša izbira; same ne dokazujejo poslovnega praga.


## 4. Korelacija s ciljno spremenljivko

Za raziskovalni cilj izberemo `ocena` in značilke razvrstimo po **absolutni**
Spearmanovi korelaciji. Predznak ohranimo, za vsako značilko izpišemo število parov.
Identifikatorjev in nominalnih kod ne vključimo samo zato, ker bi jih lahko oštevilčili.

To je retrospektiven pregled, ne izbor veljavnega modela. Pri napovedovanju ocene
že ob nakupu dejanski čas dostave in zamuda še nista znana.


In [ ]:
cilj = 'ocena'
kandidati = ['vrednost_narocila', 'postnina', 'stevilo_izdelkov', 'stevilo_prodajalcev', 'trajanje_dostave_dni', 'zamuda_dni']
korelacije = df[kandidati + [cilj]].corr(method="spearman", min_periods=30)[cilj].drop(cilj)
korelacije = korelacije.sort_values(key=lambda s: s.abs(), ascending=False)
n = pd.Series({s: len(df[[s, cilj]].dropna()) for s in kandidati}, name="znanih_parov")
display(pd.concat([korelacije.rename("Spearman"), n], axis=1).loc[korelacije.index].round(3))

fig, os = plt.subplots()
korelacije.iloc[::-1].plot.barh(ax=os)
os.axvline(0, color="black", linewidth=0.8)
os.set(xlabel=f"Spearmanov koeficient s ciljem: {cilj}", title="Prvi pregled posameznih značilk")
fig.tight_layout()
plt.show()


## 5. Primerjava skupin z `groupby`

Primerjamo ocene pravočasnih in zamujenih naročil. `size` šteje vse zapise v skupini,
`count` le znane ocene. Števili se lahko razlikujeta.

Za drug primer izberi `customer_state` in `trajanje_dostave_dni`. Na grafu pokažemo
do največji skupini z dovolj znanimi vrednostmi; celoten povzetek ostane v tabeli.


In [ ]:
skupina = 'zamuda'
meritev = 'ocena'
prag = 100
po_skupinah = df.groupby(skupina, dropna=False)[meritev].agg(
    vrstic="size", znanih="count", povprecje="mean", mediana="median"
)
display(po_skupinah.round(3))

# Omejimo samo graf; celotna tabela je izpisana zgoraj.
za_graf = po_skupinah.loc[po_skupinah["znanih"].ge(prag)].nlargest(15, "znanih").copy()
if skupina == "hnd_price":
    za_graf.index = za_graf.index.map(lambda v: f"{v:.2f}")
fig, os = plt.subplots(figsize=(10, 4))
za_graf["povprecje"].plot.bar(ax=os)
os.set(xlabel=skupina, ylabel=f"Povprečje: {meritev}",
       title=f"Največji skupini")
fig.tight_layout()
plt.show()


## 6. Kontingenčna tabela: kaj je imenovalec?

Za vsak par zamude in ocene najprej preštejemo naročila, nato primerjamo:

- **Po vrsticah (`index`):** porazdelitev ocen med pravočasnimi oziroma zamujenimi naročili.
- **Po stolpcih (`columns`):** porazdelitev zamude med naročili z določeno oceno.

V vseh tabelah uporabimo iste zapise z znanima obema vrednostma. Obe normalizaciji
sta pravilni, vendar odgovarjata na različni vprašanji.


In [ ]:
kategorija = 'zamuda'
cilj_kategorije = 'ocena'
pari_k = df[[kategorija, cilj_kategorije]].dropna()
print(f"Znanih parov: {len(pari_k):,}; izpuščenih: {len(df) - len(pari_k):,}")
stevila = pd.crosstab(pari_k[kategorija], pari_k[cilj_kategorije])
po_vrsticah = pd.crosstab(pari_k[kategorija], pari_k[cilj_kategorije], normalize="index")
po_stolpcih = pd.crosstab(pari_k[kategorija], pari_k[cilj_kategorije], normalize="columns")
print("Število zapisov:")
display(stevila)
print("Deleži po vrsticah (%):")
display((po_vrsticah * 100).round(2))
print("Deleži po stolpcih (%):")
display((po_stolpcih * 100).round(2))

fig, os = plt.subplots()
(po_vrsticah * 100).plot.bar(stacked=True, ax=os)
os.set(xlabel=kategorija, ylabel="Delež zapisov znotraj kategorije (%)",
       title="Porazdelitev znotraj posamezne kategorije", ylim=(0, 100))
os.legend(title=cilj_kategorije, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()


**Preverimo:** kateri izpis odgovori na »Kolikšen delež zamujenih naročil ima oceno 1?«
Kateri pa na »Kolikšen delež naročil z oceno 1 je zamujenih?« Zakaj številki nista enaki?


## 7. Korelacijska matrika in odvečna informacija

Dodali smo `vrednost_s_postnino = vrednost_narocila + postnina`. Zato pričakujemo
močno povezavo z vrednostjo naročila. Matrika prikaže povezave med pari, ne vzročnosti.

Visoka korelacija še ne pomeni identičnih stolpcev. Zaokrožen izpis `1,00` prav tako
ni dokaz točne enakosti. Zelo povezani vhodi lahko otežijo razlago koeficientov
linearnih modelov. Koeficienti temeljijo na parih z znanima obema vrednostma.


In [ ]:
stolpci_matrike = ['vrednost_narocila', 'vrednost_s_postnino', 'postnina', 'stevilo_izdelkov', 'stevilo_prodajalcev', 'trajanje_dostave_dni', 'zamuda_dni', 'ocena']
matrika = df[stolpci_matrike].corr(method="pearson", min_periods=30)
znane = df[stolpci_matrike].notna().astype("int64")
n_parov = znane.T.dot(znane)
display(matrika.round(3))
print("Število znanih parov za vsako kombinacijo:")
display(n_parov)

fig, os = plt.subplots(figsize=(10, 7))
slika = os.imshow(matrika, cmap="RdBu_r", vmin=-1, vmax=1)
os.set_xticks(range(len(stolpci_matrike)), labels=stolpci_matrike, rotation=45, ha="right")
os.set_yticks(range(len(stolpci_matrike)), labels=stolpci_matrike)
for vrstica in range(len(stolpci_matrike)):
    for stolpec in range(len(stolpci_matrike)):
        r = matrika.iloc[vrstica, stolpec]
        os.text(stolpec, vrstica, f"{r:.2f}", ha="center", va="center",
                color="white" if abs(r) > 0.65 else "black")
fig.colorbar(slika, ax=os, label="Pearsonov r")
os.set_title("Korelacijska matrika izbranih numeričnih stolpcev")
fig.tight_layout()
plt.show()

# Zgornji trikotnik izpusti diagonalo in dvojno štetje parov.
maska = np.triu(np.ones(matrika.shape, dtype=bool), k=1)
pari_matrike = matrika.where(maska).stack()
mocni_pari = pari_matrike.loc[pari_matrike.abs().gt(0.95)]
display(mocni_pari.rename("Pearsonov r").to_frame())


## 8. Povezanost ni vzročnost

Povezava med vrednostjo naročila in poštnino ne dokazuje, da bi podražitev istega
izdelka sama povzročila višjo poštnino. **Število izdelkov** lahko vpliva na obe vsoti;
pomembni so lahko tudi teža, razdalja, prodajalec in pogoji dostave.

Kot pregled možne skupne razlage pogledamo povezavo znotraj naročil z enim, dvema
ter tremi ali več izdelki. Kontrola enega dejavnika ne zagotovi, da se koeficient
zmanjša, niti ne odstrani vseh drugih razlag. »3 ali več« tudi ni točno enako število.


In [ ]:
primer = df[["vrednost_narocila", "postnina", "stevilo_izdelkov"]].dropna().copy()
primer["razred_izdelkov"] = pd.cut(
    primer["stevilo_izdelkov"], bins=[0, 1, 2, np.inf], labels=["1 izdelek", "2 izdelka", "3 ali več"]
)
rezultati = [{"skupina": "vsa primerljiva naročila", "n": len(primer),
              "Pearson": primer[["vrednost_narocila", "postnina"]].corr().iloc[0, 1]}]
for oznaka, podmnozica in primer.groupby("razred_izdelkov", observed=True):
    rezultati.append({"skupina": str(oznaka), "n": len(podmnozica),
                      "Pearson": podmnozica[["vrednost_narocila", "postnina"]].corr().iloc[0, 1]})
display(pd.DataFrame(rezultati).set_index("skupina").round(3))


**Za pogovor:** ali povezava znotraj skupin ostane? Ali to dokazuje vzročnost?

Opazovalni povzetki sami ne identificirajo učinka spremembe cen ali dostave.
Potrebovali bi utemeljen vzročni načrt, na primer ustrezno zasnovan randomiziran preizkus.
